In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 77 (delta 21), reused 17 (delta 17), pack-reused 48 (from 1)
Receiving objects: 100% (77/77), 1.05 MiB | 17.42 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [2]:
pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 11.5 MB/s eta 0:00:00


In [3]:
pip install ucimlrepo

In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
cdc_diabetes_health_indicators = fetch_ucirepo(id=891)

X = cdc_diabetes_health_indicators.data.features
y = cdc_diabetes_health_indicators.data.targets

print(cdc_diabetes_health_indicators.metadata)
print(cdc_diabetes_health_indicators.variables)

cdc_diabetes_data = pd.concat([X, y], axis=1)

target_col = "Diabetes_binary"

# Drop patient ID (not a feature)
if "ID" in cdc_diabetes_data.columns:
    cdc_diabetes_data = cdc_diabetes_data.drop(columns=["ID"])

# Column groups for CTABGAN / downstream models
BINARY_COLS = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
    target_col,
]
INTEGER_COLS = ["BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"]
CATEGORICAL_COLS = [col for col in BINARY_COLS if col in cdc_diabetes_data.columns]

# Ensure numeric types (dataset is already numeric; this is a safety step)
for col in cdc_diabetes_data.columns:
    cdc_diabetes_data[col] = pd.to_numeric(cdc_diabetes_data[col], errors="coerce")
    cdc_diabetes_data[col] = cdc_diabetes_data[col].fillna(cdc_diabetes_data[col].median())

# Positive class for binary metrics (1 = diabetes / prediabetes)
pos_label = 1

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Fast dev mode: fewer epochs for generators only (set False for full paper run)
FAST_MODE = True
CTABGAN_EPOCHS = 10 if FAST_MODE else 150
WGAN_EPOCHS = 10 if FAST_MODE else 100
SDV_EPOCHS = 10 if FAST_MODE else 300

# Classifier evaluation always uses 10 seeds (same as cancer notebook)
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

# All 6 synthetic generators for TSTR evaluation
GENERATORS_TO_EVAL = [
    "CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "WGAN_GP", "CTABGAN"
]

# Stratified subsample (full dataset has 253680 rows)
_, cdc_diabetes_data = train_test_split(
    cdc_diabetes_data,
    train_size=N_SAMPLES,
    stratify=cdc_diabetes_data[target_col],
    random_state=SEED,
)
cdc_diabetes_data = cdc_diabetes_data.reset_index(drop=True)

X = cdc_diabetes_data.drop(columns=[target_col])
y = cdc_diabetes_data[target_col]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(cdc_diabetes_data)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}
synthetic_datasets = {}
quality_results = []

{'uci_id': 891, 'name': 'CDC Diabetes Health Indicators', 'repository_url': 'https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators', 'data_url': 'https://archive.ics.uci.edu/static/public/891/data.csv', 'abstract': 'The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy. ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 253680, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Sex', 'Age', 'Education Level', 'Income'], 'target_col': ['Diabetes_binary'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_

In [5]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# GENERATOR TRAINING DATA (no stratified split)
# ---------------------------------------------------

# Generators: use all subsampled data (no stratified split here)
train_real = cdc_diabetes_data.copy()
test_real = cdc_diabetes_data.copy()

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# CTABGAN
# ---------------------------------------------------

try:

    data_path = "cdc_diabetes_health_indicators_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=CATEGORICAL_COLS,
        log_columns=[],
        mixed_columns={},
        integer_columns=INTEGER_COLS,
        problem_type={"Classification": target_col}
    )

    ctabgan.synthesizer.epochs = CTABGAN_EPOCHS
    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    for col in CATEGORICAL_COLS:
        if col in synthetic_ctabgan.columns:
            synthetic_ctabgan[col] = (
                pd.to_numeric(synthetic_ctabgan[col], errors="coerce")
                .fillna(train_real[col].mode()[0])
                .round()
                .clip(0, 1)
                .astype(int)
            )

    for col in INTEGER_COLS:
        if col in synthetic_ctabgan.columns:
            col_min = int(train_real[col].min())
            col_max = int(train_real[col].max())
            synthetic_ctabgan[col] = (
                pd.to_numeric(synthetic_ctabgan[col], errors="coerce")
                .fillna(train_real[col].median())
                .round()
                .clip(col_min, col_max)
                .astype(int)
            )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 10/10 [29:22<00:00, 176.23s/it]


Finished training in 1880.3116545677185  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 35.67it/s]|
Column Shapes Score: 89.13%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:02<00:00, 88.97it/s]|
Column Pair Trends Score: 76.59%

Overall Score (Average): 82.86%

CTABGAN: 0.8286


In [6]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(WGAN_EPOCHS):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_wgan[target_col] = encoder.inverse_transform(
        synthetic_wgan[target_col]
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:01<00:00, 20.59it/s]|
Column Shapes Score: 17.54%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:03<00:00, 69.09it/s]|
Column Pair Trends Score: -0.0%

Overall Score (Average): 8.77%

WGAN_GP: 0.0877


In [7]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "TVAE": TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 34.91it/s]|
Column Shapes Score: 91.82%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:02<00:00, 86.08it/s]|
Column Pair Trends Score: 85.44%

Overall Score (Average): 88.63%

CTGAN: 0.8863
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 33.81it/s]|
Column Shapes Score: 89.99%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:03<00:00, 67.74it/s]|
Column Pair Trends Score: 81.75%

Overall Score (Average): 85.87%

CopulaGAN: 0.8587
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 36.40it/s]|
Column Shapes Score: 92.9%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:03<00:00, 70.78it/s]|
Column Pair Trends Score: 84.56%

Overall Score (Average): 88.73%

TVAE: 0.8873
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 34.22it/s]|
Column Shapes Sc

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# 10 seeds for TRTR / TSTR evaluation (same as cancer notebook)
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

# All 10 classifiers. FAST_MODE uses cheaper equivalents for slow models (esp. SVM-RBF).
if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        # LinearSVC ~100x faster than RBF SVC; keep name for result tables
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(
            kernel="rbf", cache_size=1000, tol=1e-3, random_state=42
        ),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")

Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd


In [10]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
# This duplicate cell was removed — SVM-RBF (probability=True) caused multi-hour runs.
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_train),
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_test),
            )

            scaler = StandardScaler().fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)
            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train_s, y_train)
            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [12]:
import pandas as pd

label_col = "Diabetes_binary"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real)")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Generators: {len(model_order)}"
)
print(f"Classifier models: {list(models.keys())}")
print(f"Synthetic generators: {model_order}")

trtr_results = evaluate_models(
    train_df=cdc_diabetes_data,
    test_df=cdc_diabetes_data,
    label="Diabetes_binary",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=cdc_diabetes_data,
        label="Diabetes_binary",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)
Classifiers: 10 | Seeds: 10 | Generators: 6
Classifier models: ['LogReg', 'SVM-RBF', 'KNN', 'NaiveBayes', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoost', 'AdaBoost', 'MLP']
Synthetic generators: ['CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'WGAN_GP', 'CTABGAN']
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
7,GradientBoost,0.8654 ± 0.0005,0.1723 ± 0.0041,0.6012 ± 0.0116,0.1006 ± 0.0027
9,MLP,0.8645 ± 0.0013,0.2639 ± 0.0297,0.5446 ± 0.0199,0.1755 ± 0.0261
1,SVM-RBF,0.8636 ± 0.0004,0.1218 ± 0.0037,0.5911 ± 0.0115,0.0679 ± 0.0022
0,LogReg,0.8635 ± 0.0007,0.2389 ± 0.0046,0.5360 ± 0.0093,0.1537 ± 0.0031
8,AdaBoost,0.8631 ± 0.0006,0.2907 ± 0.0116,0.5229 ± 0.0066,0.2015 ± 0.0114
5,RandomForest,0.8586 ± 0.0009,0.2543 ± 0.0041,0.4792 ± 0.0086,0.1731 ± 0.0030
4,DecisionTree,0.8577 ± 0.0011,0.2518 ± 0.0090,0.4710 ± 0.0100,0.1720 ± 0.0082
6,ExtraTrees,0.8513 ± 0.0009,0.2488 ± 0.0051,0.4204 ± 0.0070,0.1768 ± 0.0046
2,KNN,0.8472 ± 0.0009,0.2759 ± 0.0037,0.4063 ± 0.0048,0.2089 ± 0.0041
3,NaiveBayes,0.7729 ± 0.0016,0.4111 ± 0.0023,0.3219 ± 0.0019,0.5689 ± 0.0058


CTGAN - TSTR
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
1,SVM-RBF,0.8612 ± 0.0002,0.0472 ± 0.0162,0.5484 ± 0.0202,0.0248 ± 0.0089
0,LogReg,0.8610 ± 0.0005,0.0985 ± 0.0124,0.5102 ± 0.0171,0.0546 ± 0.0075
8,AdaBoost,0.8605 ± 0.0005,0.0924 ± 0.0310,0.4962 ± 0.0153,0.0516 ± 0.0188
5,RandomForest,0.8592 ± 0.0004,0.0543 ± 0.0125,0.4228 ± 0.0233,0.0291 ± 0.0071
7,GradientBoost,0.8590 ± 0.0010,0.0548 ± 0.0174,0.4148 ± 0.0446,0.0295 ± 0.0099
6,ExtraTrees,0.8535 ± 0.0012,0.1071 ± 0.0142,0.3541 ± 0.0271,0.0632 ± 0.0092
2,KNN,0.8518 ± 0.0013,0.1623 ± 0.0128,0.3828 ± 0.0103,0.1032 ± 0.0103
9,MLP,0.8517 ± 0.0024,0.1686 ± 0.0132,0.3868 ± 0.0159,0.1082 ± 0.0116
3,NaiveBayes,0.8149 ± 0.0057,0.3983 ± 0.0100,0.3646 ± 0.0072,0.4405 ± 0.0285
4,DecisionTree,0.8023 ± 0.0106,0.2459 ± 0.0219,0.2637 ± 0.0180,0.2326 ± 0.0332


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
1,SVM-RBF,0.8569 ± 0.0015,0.3063 ± 0.0078,0.4720 ± 0.0100,0.2269 ± 0.0091
7,GradientBoost,0.8551 ± 0.0018,0.2705 ± 0.0130,0.4539 ± 0.0133,0.1932 ± 0.0142
0,LogReg,0.8548 ± 0.0013,0.3302 ± 0.0073,0.4622 ± 0.0075,0.2570 ± 0.0092
5,RandomForest,0.8527 ± 0.0016,0.2742 ± 0.0140,0.4376 ± 0.0108,0.2001 ± 0.0147
8,AdaBoost,0.8519 ± 0.0036,0.3444 ± 0.0119,0.4509 ± 0.0180,0.2795 ± 0.0187
6,ExtraTrees,0.8455 ± 0.0016,0.2912 ± 0.0076,0.4037 ± 0.0083,0.2279 ± 0.0090
2,KNN,0.8338 ± 0.0025,0.2969 ± 0.0068,0.3617 ± 0.0092,0.2520 ± 0.0081
9,MLP,0.8302 ± 0.0022,0.3353 ± 0.0118,0.3688 ± 0.0076,0.3078 ± 0.0169
3,NaiveBayes,0.7851 ± 0.0073,0.4149 ± 0.0059,0.3346 ± 0.0059,0.5473 ± 0.0274
4,DecisionTree,0.7316 ± 0.0164,0.2801 ± 0.0121,0.2243 ± 0.0127,0.3746 ± 0.0243


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,GradientBoost,0.010276,-0.098252,0.147333,-0.092586,0.8654 ± 0.0005,0.8551 ± 0.0018
1,CTABGAN,MLP,0.034277,-0.071412,0.175735,-0.132282,0.8645 ± 0.0013,0.8302 ± 0.0022
2,CTABGAN,SVM-RBF,0.006718,-0.184487,0.119089,-0.158997,0.8636 ± 0.0004,0.8569 ± 0.0015
3,CTABGAN,LogReg,0.008764,-0.091286,0.073784,-0.103281,0.8635 ± 0.0007,0.8548 ± 0.0013
4,CTABGAN,AdaBoost,0.011176,-0.053668,0.072015,-0.077986,0.8631 ± 0.0006,0.8519 ± 0.0036
5,CTABGAN,RandomForest,0.005899,-0.019937,0.041648,-0.026999,0.8586 ± 0.0009,0.8527 ± 0.0016
6,CTABGAN,DecisionTree,0.126128,-0.028296,0.246708,-0.202642,0.8577 ± 0.0011,0.7316 ± 0.0164
7,CTABGAN,ExtraTrees,0.005851,-0.042357,0.016713,-0.051143,0.8513 ± 0.0009,0.8455 ± 0.0016
8,CTABGAN,KNN,0.013464,-0.020973,0.044610,-0.043019,0.8472 ± 0.0009,0.8338 ± 0.0025
9,CTABGAN,NaiveBayes,-0.012156,-0.003811,-0.012750,0.021602,0.7729 ± 0.0016,0.7851 ± 0.0073


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
1,CTGAN,0.003269,0.110026,0.075022,0.086154
5,WGAN_GP,0.006588,0.047764,0.033847,0.026171
3,GaussianCopula,0.010459,0.183897,0.255751,0.139770
0,CTABGAN,0.021040,-0.061448,0.092488,-0.086733
4,TVAE,0.021432,-0.062311,0.102588,-0.093218
2,CopulaGAN,0.033001,-0.076803,0.130385,-0.127517


In [13]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")


Results saved to: TRTR_TSTR_results.xlsx
